# Interfaces and Interchangeable Components


```{contents}
:local:
:depth: 2
```


```{index} Depend on the Capability the Client Needs
```



```{index} Depend on the Capability the Client Needs
```

## Depend on the Capability the Client Needs



A checkout coordinator needs a way to request payment. It need not know every field or helper method of a particular payment provider. An interface names the small capability the client relies on. An implementing class can have a different internal representation and still satisfy that capability, provided its behavior and failure policy meet the contract.

Supplying the collaborator through a constructor makes the dependency visible. The coordinator can reject a missing required collaborator at construction rather than failing unexpectedly during a later transaction. This is an application of composition and null contracts, not a requirement to use an elaborate dependency-injection framework.

A test double can record a requested amount and return a prepared response. That permits a test of the coordinator's interaction without contacting a real payment service. Check more than whether any call occurred: the amount, number of calls, result interpretation and state on rejection can all matter. A recording double observes collaboration; it does not by itself prove that the real external service works.

Guided design: a report formatter needs only a way to produce a line of text. Compare making every formatter inherit a large invoice-processing base class with supplying a small formatting capability. State which parts are actually shared and which are accidental similarities. An interface does not imply that all implementations share state, a base implementation or the same operational cost.

For related explanation, see the further reading below.[^development-reading]


A checkout needs a discount calculation, not necessarily a family of classes sharing fields. An interface gives that requirement a name. A class can implement the capability while keeping its own representation and constructor design.

Connect the idea to Chapter 11: `Checkout` has a discount component. Composition describes the relationship; an interface describes the operations that the component promises. These techniques work together rather than competing for a single “best” programming style.


```{index} A Small Capability Contract
```

## A Small Capability Contract




```{index} A Small Capability Contract
```


The `IDiscountPolicy` interface declares one operation. Implementations provide a public method with the required signature. They do not use `override` merely to implement this interface. The `I` naming prefix is a convention that makes interface types recognizable.

The exercise contract additionally requires nonnegative input, rejection of negative subtotals, and a result between zero and the subtotal. Those business rules need explanation and checks; the interface declaration alone cannot express all of them.


In [1]:
using System;

IDiscountPolicy policy = new PercentageDiscount(0.10m);
Console.WriteLine(policy.Apply(100m).ToString("F2"));

interface IDiscountPolicy
{
    decimal Apply(decimal subtotal);
}

class PercentageDiscount : IDiscountPolicy
{
    private readonly decimal rate;
    public PercentageDiscount(decimal rate)
    {
        if (rate < 0m || rate > 1m) throw new ArgumentOutOfRangeException(nameof(rate));
        this.rate = rate;
    }
    public decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal * (1m - rate);
    }
}


90.00


An interface is not simply an abstract class with a different name. It describes a capability without requiring a shared instance-field representation or constructor. A class may implement several interfaces but inherits directly from at most one class. This chapter uses only method signatures; modern C# supports additional interface features that are unnecessary here.

Keep contracts small enough that implementations can satisfy them coherently. A single interface requiring discounts, database persistence, console menus, and stock reservations would combine unrelated responsibilities.


```{index} Supply Collaborators Through Constructors
```

## Supply Collaborators Through Constructors




```{index} Supply Collaborators Through Constructors
```


`Checkout` receives an `IDiscountPolicy` in its constructor and stores the reference. The caller chooses the implementation. This is **constructor injection**: supplying a dependency explicitly instead of hard-coding its creation inside the client. It does not require a framework or a dependency-injection container.

Because checkout only calls the interface, a new policy does not require editing its calculation method. The model still needs validation at the right boundary; injection is not a substitute for a behavioral contract.


In [1]:
using System;

Checkout ordinary = new Checkout(new NoDiscount());
Checkout member = new Checkout(new PercentageDiscount(0.10m));
Console.WriteLine(ordinary.Quote(100m).ToString("F2"));
Console.WriteLine(member.Quote(100m).ToString("F2"));

interface IDiscountPolicy
{
    decimal Apply(decimal subtotal);
}

class NoDiscount : IDiscountPolicy
{
    public decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class PercentageDiscount : IDiscountPolicy
{
    private readonly decimal rate;
    public PercentageDiscount(decimal rate)
    {
        if (rate < 0m || rate > 1m) throw new ArgumentOutOfRangeException(nameof(rate));
        this.rate = rate;
    }
    public decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal * (1m - rate);
    }
}

class Checkout
{
    private readonly IDiscountPolicy discount;
    public Checkout(IDiscountPolicy discount)
    {
        if (discount is null) throw new ArgumentNullException(nameof(discount));
        this.discount = discount;
    }
    public decimal Quote(decimal subtotal) { return discount.Apply(subtotal); }
}


100.00
90.00


The two checkout objects share the same client implementation and use different components. They do not inherit from separate “ordinary checkout” and “member checkout” classes. If discount selection changes at a business boundary, constructor creation can choose a different component without multiplying checkout subtypes.

A field declared `readonly` prevents the checkout from reassigning that collaborator after construction. It does not guarantee that the supplied implementation has no mutable fields. State any side-effect expectations and test the implementations you accept.


```{index} Use a Test Double to Observe Collaboration
```

## Use a Test Double to Observe Collaboration




```{index} Use a Test Double to Observe Collaboration
```


A **test double** is a deliberately simple substitute used to observe a collaboration. Here a recording policy counts calls and returns the input unchanged. It satisfies the result contract and makes delegation visible without relying on a particular percentage formula.

Check that checkout passes the actual subtotal and invokes the collaborator exactly once. These observations would reveal a client that ignores its dependency, calls it twice, or supplies the wrong amount.


In [1]:
using System;

RecordingDiscount recording = new RecordingDiscount();
Checkout checkout = new Checkout(recording);
decimal result = checkout.Quote(40m);
Console.WriteLine($"Calls: {recording.Calls}");
Console.WriteLine($"Input: {recording.LastSubtotal:F2}");
Console.WriteLine($"Result: {result:F2}");

interface IDiscountPolicy
{
    decimal Apply(decimal subtotal);
}

class Checkout
{
    private readonly IDiscountPolicy discount;
    public Checkout(IDiscountPolicy discount)
    {
        if (discount is null) throw new ArgumentNullException(nameof(discount));
        this.discount = discount;
    }
    public decimal Quote(decimal subtotal) { return discount.Apply(subtotal); }
}

class RecordingDiscount : IDiscountPolicy
{
    public int Calls { get; private set; }
    public decimal LastSubtotal { get; private set; }
    public decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        Calls++;
        LastSubtotal = subtotal;
        return subtotal;
    }
}


Calls: 1
Input: 40.00
Result: 40.00


A call-count check is useful here because delegation is the behavior under test. Do not assert every private helper call in an application: that would make harmless refactoring difficult. Prefer checks of observable results, with targeted collaboration checks where they clarify the contract.


```{index} Choose Composition, an Interface, or a Base Class
```

## Choose Composition, an Interface, or a Base Class




```{index} Choose Composition, an Interface, or a Base Class
```


Start with the smallest design that supports the requirements. A concrete collaborator is enough when there is one implementation. Introduce an interface when clients need interchangeable capabilities or a clear testing boundary. Use a base class when the types share a meaningful substitution relationship and common behavior or state.

| Need | Design to consider |
| --- | --- |
| Order owns several lines. | Composition with a concrete line type. |
| Checkout supports several discount calculations. | Composition through a small interface. |
| Several payment kinds share validated amount state. | An abstract base class with overrides. |
| One calculation repeated in one class. | A private helper method. |

Do not create an interface for every class automatically. An interface adds another contract to maintain. Likewise, do not build a deep hierarchy to anticipate features that do not exist yet. Use an explicit use case and a short test to justify an extension point.

In Chapter 13, interfaces become contracts for abstract data types, and generics let one implementation support multiple element types. The connection is behavioral: clients rely on an operation's promise while representation details remain hidden. For persistence after this chapter, use the optional database appendix.


```{index} Practice: Replace a Component Without Changing Its Client
```

## Practice: Replace a Component Without Changing Its Client




```{index} Practice: Replace a Component Without Changing Its Client
```


Complete `NoDiscount` so it preserves valid subtotals and rejects negative input. Use the existing checkout unchanged.


In [ ]:
using System;

Checkout checkout = new Checkout(new NoDiscount());
Console.WriteLine(checkout.Quote(45m).ToString("F2"));

interface IDiscountPolicy
{
    decimal Apply(decimal subtotal);
}

class NoDiscount : IDiscountPolicy
{
    public decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        // Your code starts here.
        return 0m; // TODO: preserve the subtotal.
        // Your code ends here.
    }
}

class Checkout
{
    private readonly IDiscountPolicy discount;
    public Checkout(IDiscountPolicy discount)
    {
        if (discount is null) throw new ArgumentNullException(nameof(discount));
        this.discount = discount;
    }
    public decimal Quote(decimal subtotal) { return discount.Apply(subtotal); }
}


In [1]:
using System;

Checkout checkout = new Checkout(new NoDiscount());
Console.WriteLine(checkout.Quote(45m).ToString("F2"));

interface IDiscountPolicy
{
    decimal Apply(decimal subtotal);
}

class NoDiscount : IDiscountPolicy
{
    public decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class Checkout
{
    private readonly IDiscountPolicy discount;
    public Checkout(IDiscountPolicy discount)
    {
        if (discount is null) throw new ArgumentNullException(nameof(discount));
        this.discount = discount;
    }
    public decimal Quote(decimal subtotal) { return discount.Apply(subtotal); }
}


45.00


```{rubric} Footnotes
```


[^1203_interfaces-1]: This section uses ordinary interface methods. Default interface implementations, static abstract members, and generic variance are later language topics, not prerequisites for interchangeable components.


[^development-reading]: Further reading: [Microsoft, Methods in C#](https://learn.microsoft.com/en-us/dotnet/csharp/methods). The worked business scenarios and guided reasoning here are original teaching examples.
